This project optimizes research-paper retrieval under a priority hierarchy:
1. semantic relevance is mandatory,
2. recency is preferred when relevance is comparable,
3. peer-reviewed/source-quality signals are used as a secondary tie-breaker.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.retriever import ResearchPaperRetriever

# Phase 4: Automated RAG Evaluation & Benchmarking

In this notebook, we benchmark our metadata-aware research RAG pipeline using the synthetic evaluation dataset (`eval_df`) constructed in Phase 1. The evaluation is designed to isolate retrieval behavior from downstream LLM generation, allowing us to directly measure the impact of metadata-aware reranking on semantic retrieval quality, freshness, and publication maturity.

Unlike traditional single-document retrieval benchmarks, our evaluation uses **group-based relevance labels**. Each synthetic query is associated with a small semantic neighborhood of related papers derived from full-corpus retrieval, reflecting the reality that research queries often have multiple valid or partially valid answers rather than one exact ground-truth document.

The evaluation benchmark is intentionally constructed as a **recency-aware retrieval setting**. Seed papers are sampled from recent published papers (2025–2026), while retrieval is performed over the full research corpus. This framing aligns with the project objective: helping researchers surface current and publication-mature literature without sacrificing semantic relevance.

## 1. Multi-Objective Ranking Validation

We evaluate the retrieval behavior of our metadata-aware RAG pipeline against a pure semantic-search baseline. The benchmark combines standard information retrieval metrics with metadata-oriented retrieval diagnostics tailored to our research-assistant objective.

### Retrieval Metrics

- **Group Hit@10**: proportion of queries where at least one paper from the relevant semantic group appears in the top-10 results
- **Group Recall@10**: proportion of the relevant paper group retrieved in the top-10 results
- **Group MRR@10**: reciprocal rank of the first retrieved paper belonging to the relevant semantic group
- **Seed Hit@10**: whether the original seed paper used to generate the query appears in the top-10 results

### Metadata-Oriented Metrics

- **Freshness@10**: average publication year of the top-10 retrieved papers
- **Published Rate@10**: proportion of top-10 retrieved papers with a publication signal (`doi` or `journal-ref`)

Together, these metrics allow us to evaluate not only semantic retrieval quality, but also whether the system successfully surfaces newer and publication-mature research papers.

### Methodology (Relevance-First Metadata Reranking)

Raw vector similarity serves as the semantic anchor of the retrieval system and defines the upper bound for pure meaning-based search. Our reranker then injects two secondary metadata signals:

- **Recency** (`w_recency`) to prefer newer research
- **Publication signal** (`w_publication`) to prioritize publication-mature papers

The ranking objective follows a strict priority hierarchy:

$$
\text{Semantic Relevance}
\; > \;
\text{Recency}
\; > \;
\text{Publication Signal}
$$

Metadata is therefore intended to *modulate* rankings among already relevant candidates, not override semantic similarity itself. This distinction is important: the goal is not simply to maximize retrieval metrics, but to produce retrieval results better aligned with real-world research workflows, where researchers often prefer newer and publication-mature papers when topical relevance is comparable.

### Synthetic Query Construction & Leakage Mitigation

While the benchmark uses synthetic queries, the evaluation reduces direct document leakage by generating queries from paper titles and condensed summaries rather than full abstracts. This prevents the retrieval system from exploiting highly specific abstract phrasing or rare lexical artifacts copied directly into the query generation process.

Additionally, relevance is evaluated against semantic paper groups rather than a single exact document, creating a more realistic approximation of real-world research retrieval behavior where multiple papers may validly satisfy the same information need.

In [2]:
# Load previously saved eval_df
eval_df = pd.read_parquet("../data/processed/arxiv_eval_dataset.parquet")
eval_df.head()

,query,seed_paper_id,relevant_paper_ids,seed_title,seed_summary,seed_abstract,year,is_published,categories
0,What are the most effective federated learning...,arxiv.2502.17226,"[arxiv.2502.17226, arxiv.2103.11870, arxiv.191...",Electrical Load Forecasting over Multihop Smar...,The research investigates improving electric l...,Electric load forecasting is essential for pow...,2025,True,None
1,How can visual representations of vector graph...,arxiv.2404.06479,"[arxiv.2404.06479, arxiv.2604.07518, arxiv.250...",Visually Descriptive Language Model for Vector...,Research investigates the challenge of large m...,"Despite significant advancements, large multim...",2025,True,None
2,What are the key terrain features and their re...,arxiv.2309.15400,"[arxiv.2309.15400, arxiv.2604.04339, arxiv.250...",Interpretable AI-Driven Discovery of Terrain-P...,The research investigates the lack of interpre...,Despite the remarkable strides made by AI-driv...,2025,True,None
3,How can diffusion models effectively enhance t...,arxiv.2503.11181,"[arxiv.2503.11181, arxiv.2512.05139, arxiv.221...",Multi-Stage Generative Upscaler: Reconstructin...,This research addresses the challenge of impro...,The reconstruction of low-resolution football ...,2026,True,None
4,What machine learning techniques can effective...,arxiv.2008.12065,"[arxiv.2008.12065, arxiv.2506.19789, arxiv.241...",Propensity-to-Pay: Machine Learning for Estima...,The research investigates the use of machine l...,Predicting a customer's propensity-to-pay at a...,2025,True,None


In [19]:
eval_df["year"].value_counts(normalize=True).mul(100).round(2)

year
2025    68.5
2026    31.5
Name: proportion, dtype: float64

In [21]:
eval_df["is_published"].value_counts(normalize=True).mul(100).round(2)

is_published
True    100.0
Name: proportion, dtype: float64

### Experiments Configurations

In [23]:
WEIGHT_CONFIGS = {
    "Baseline": {
        "w_vector": 1.00,
        "w_recency": 0.00,
        "w_publication": 0.00,
    },
    "Light Metadata Boost": {
        "w_vector": 0.90,
        "w_recency": 0.05,
        "w_publication": 0.05,
    },
    "Light Recency Boost": {
        "w_vector": 0.85,
        "w_recency": 0.10,
        "w_publication": 0.05,
    },
    "Balanced": {
        "w_vector": 0.80,
        "w_recency": 0.10,
        "w_publication": 0.10,
    },
    "Equalized": {
        "w_vector": 0.33,
        "w_recency": 0.33,
        "w_publication": 0.33,
    },
    "Aggressive Freshness Bias": {
        "w_vector": 0.05,
        "w_recency": 0.90,
        "w_publication": 0.05,
    },
    "Aggressive Publication Bias": {
        "w_vector": 0.05,
        "w_recency": 0.05,
        "w_publication": 0.90,
    },
}

### Metric helpers

In [7]:
def group_hit_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    return int(any(doc_id in relevant_ids for doc_id in retrieved_ids[:k]))


def group_recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    if not relevant_ids:
        return 0.0

    retrieved_top_k = set(retrieved_ids[:k])
    hits = retrieved_top_k.intersection(relevant_ids)

    return len(hits) / len(relevant_ids)


def reciprocal_rank_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def seed_hit_at_k(retrieved_ids, seed_id, k):
    return int(seed_id in retrieved_ids[:k])


def freshness_at_k(years, k):
    years = [y for y in years[:k] if pd.notna(y)]
    return np.mean(years) if years else np.nan


def published_rate_at_k(is_published_values, k):
    values = [int(v) for v in is_published_values[:k] if pd.notna(v)]
    return np.mean(values) if values else np.nan

### Initialize retriever

In [8]:
retriever = ResearchPaperRetriever()
retriever.load_index()

### Run one configuration

In [13]:
def evaluate_single_config(
    retriever,
    eval_df,
    config_name,
    weights,
    k=10,
    fetch_k=30,
):
    """
    Evaluate one retrieval configuration using group-based relevance labels.

    Each query has:
    - seed_paper_id: original paper used to generate the query
    - relevant_paper_ids: semantic neighborhood considered relevant
    """

    retriever.w_vector = weights["w_vector"]
    retriever.w_recency = weights["w_recency"]
    retriever.w_publication = weights["w_publication"]

    group_hits = []
    group_recalls = []
    reciprocal_ranks = []
    seed_hits = []
    freshness_scores = []
    published_rates = []

    logs = []

    for idx, row in tqdm(
        eval_df.iterrows(),
        total=len(eval_df),
        desc=f"Evaluating {config_name}",
    ):
        query = row["query"]
        seed_id = row["seed_paper_id"]
        relevant_ids = row["relevant_paper_ids"]

        # Defensive handling if loaded from parquet/stringified lists
        if isinstance(relevant_ids, str):
            import ast
            relevant_ids = ast.literal_eval(relevant_ids)

        results = retriever.retrieve(
            query=query,
            k=k,
            fetch_k=fetch_k,
            rerank=True,
        )

        retrieved_ids = [r.get("doc_id") or r.get("id") for r in results]
        retrieved_years = [r["year"] for r in results]
        retrieved_is_published = [r["is_published"] for r in results]

        group_hits.append(group_hit_at_k(retrieved_ids, relevant_ids, k))
        group_recalls.append(group_recall_at_k(retrieved_ids, relevant_ids, k))
        reciprocal_ranks.append(reciprocal_rank_at_k(retrieved_ids, relevant_ids, k))
        seed_hits.append(seed_hit_at_k(retrieved_ids, seed_id, k))
        freshness_scores.append(freshness_at_k(retrieved_years, k))
        published_rates.append(published_rate_at_k(retrieved_is_published, k))

        logs.append({
            "config_name": config_name,
            "query_id": f"Q_{idx:03d}",
            "query": query,
            "seed_paper_id": seed_id,
            "relevant_paper_ids": relevant_ids,
            "retrieved_ids": retrieved_ids,
            "retrieved_years": retrieved_years,
            "retrieved_is_published": retrieved_is_published,
            "group_hit": group_hits[-1],
            "group_recall": group_recalls[-1],
            "reciprocal_rank": reciprocal_ranks[-1],
            "seed_hit": seed_hits[-1],
        })

    summary = {
        "Set Up": config_name,
        "w_vector / w_recency / w_publication": (
            f"{weights['w_vector']:.2f} / "
            f"{weights['w_recency']:.2f} / "
            f"{weights['w_publication']:.2f}"
        ),
        f"Group Hit@{k}": np.mean(group_hits),
        f"Group Recall@{k}": np.mean(group_recalls),
        f"Group MRR@{k}": np.mean(reciprocal_ranks),
        f"Seed Hit@{k}": np.mean(seed_hits),
        f"Freshness@{k} (Avg Year)": np.nanmean(freshness_scores),
        f"Published Rate@{k}": np.nanmean(published_rates),
    }

    return summary, pd.DataFrame(logs)

### Run all configurations

In [14]:
def evaluate_config_grid(
    retriever,
    eval_df,
    weight_configs,
    k=10,
    fetch_k=30,
):
    summaries = []
    all_logs = []

    for config_name, weights in weight_configs.items():
        summary, logs_df = evaluate_single_config(
            retriever=retriever,
            eval_df=eval_df,
            config_name=config_name,
            weights=weights,
            k=k,
            fetch_k=fetch_k,
        )

        summaries.append(summary)
        all_logs.append(logs_df)

    summary_df = pd.DataFrame(summaries)
    logs_df = pd.concat(all_logs, ignore_index=True)

    return summary_df, logs_df

In [24]:
performance_matrix_df, eval_logs_df = evaluate_config_grid(
    retriever=retriever,
    eval_df=eval_df,
    weight_configs=WEIGHT_CONFIGS,
    k=10,
    fetch_k=30,
)

performance_matrix_df

Evaluating Baseline: 100%|█| 400/400 [01:38<00:00,  4
Evaluating Light Metadata Boost: 100%|█| 400/400 [01:
Evaluating Light Recency Boost: 100%|█| 400/400 [02:1
Evaluating Balanced: 100%|█| 400/400 [01:48<00:00,  3
Evaluating Equalized: 100%|█| 400/400 [01:48<00:00,  
Evaluating Aggressive Freshness Bias: 100%|█| 400/400
Evaluating Aggressive Publication Bias: 100%|█| 400/4


,Set Up,w_vector / w_recency / w_publication,Group Hit@10,Group Recall@10,Group MRR@10,Seed Hit@10,Freshness@10 (Avg Year),Published Rate@10
0,Baseline,1.00 / 0.00 / 0.00,0.9825,0.39850,0.888937,0.9275,2023.69800,0.24325
1,Light Metadata Boost,0.90 / 0.05 / 0.05,0.9825,0.39575,0.898187,0.9400,2023.83825,0.29025
2,Light Recency Boost,0.85 / 0.10 / 0.05,0.9850,0.39675,0.898100,0.9425,2024.02200,0.29100
3,Balanced,0.80 / 0.10 / 0.10,0.9825,0.38775,0.904319,0.9450,2023.99250,0.36250
4,Equalized,0.33 / 0.33 / 0.33,0.9925,0.31075,0.946256,0.9650,2024.41025,0.54350
5,Aggressive Freshness Bias,0.05 / 0.90 / 0.05,0.9875,0.27950,0.657725,0.9575,2025.31225,0.26825
6,Aggressive Publication Bias,0.05 / 0.05 / 0.90,0.9925,0.30675,0.947177,0.9650,2024.32175,0.56200


## 2. Evaluation Design Considerations & Limitations

### Recency-Aware Benchmark Bias

The benchmark is intentionally biased toward recent published research because the project itself targets recency-aware research retrieval. As a result, metadata-heavy configurations are naturally advantaged on metrics such as Seed Hit@10 and Group MRR@10.

This bias is therefore not accidental, but aligned with the operational objective of the system: surfacing recent and publication-mature research when semantic relevance is comparable.

### Retrieval Tradeoffs Under Metadata Reranking

Despite this benchmark bias, the evaluation still exposes meaningful retrieval tradeoffs. Configurations with aggressive metadata weighting improve freshness and publication maturity, but often reduce Group Recall@10, indicating a loss of broader semantic coverage.

Empirically, moderate metadata weighting produced the most stable retrieval behavior. The Balanced configuration `(0.80 / 0.10 / 0.10)` improved Group MRR@10, Seed Hit@10, Freshness@10, and Published Rate@10 over the pure semantic baseline while preserving most group-level semantic coverage.

In contrast, heavily metadata-driven configurations achieved stronger ranking alignment with recent published papers, but at the cost of substantially lower Group Recall@10, suggesting over-prioritization of metadata signals relative to semantic diversity.

### Production Considerations

From a production perspective, these results suggest that metadata signals should act as lightweight ranking modifiers rather than dominant retrieval objectives. Semantic similarity remains the primary retrieval anchor, while recency and publication maturity help refine ranking quality among already relevant candidates.

The experiments also demonstrate that overly aggressive metadata weighting can reduce topical breadth, potentially suppressing foundational or semantically relevant papers that fall outside recent or publication-mature subsets. In practical deployment, moderate reranking weights therefore provide the best balance between semantic fidelity, freshness, and research reliability.